In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

# dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
dense_index = DenseIndex(model, "../data/processed/_dense_court_removed_citation", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

DenseIndex.embeddings:  (2099335, 1024)


In [4]:
test_df = pd.read_csv("../data/test_rewrite_001.csv")
test_id_2_query_d = {}
for query_id, query in zip(test_df['query_id'], test_df['query']):
    test_id_2_query_d[query_id] = query

In [5]:
import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import colbert_utils

test_multi_question_df = pd.read_csv("../data/test_rewrite_split_question_001.csv")

recall_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(test_multi_question_df['query_id'], test_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in test_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

query_id_2_recall_hits = {}

for query_id, query_list in query_id_2_query_list.items():
    raw_query = query_list[-1]

    hits1 = dense_index.search_with_score(raw_query, top_k=1000)
    hits2 = sparse_index.search_with_score(raw_query, top_k=1000)
    all_hits = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
    
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    query_id_2_recall_hits[query_id] = all_hits.copy()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[test_001] hits_merge.len: 1906
[test_002] hits_merge.len: 1873
[test_003] hits_merge.len: 1750
[test_004] hits_merge.len: 1745
[test_005] hits_merge.len: 1771
[test_006] hits_merge.len: 1947
[test_007] hits_merge.len: 1880
[test_008] hits_merge.len: 1582
[test_009] hits_merge.len: 1857
[test_010] hits_merge.len: 1903
[test_011] hits_merge.len: 1936
[test_012] hits_merge.len: 1901
[test_013] hits_merge.len: 1849
[test_014] hits_merge.len: 1760
[test_015] hits_merge.len: 1649
[test_016] hits_merge.len: 1637
[test_017] hits_merge.len: 1846
[test_018] hits_merge.len: 1958
[test_019] hits_merge.len: 1854
[test_020] hits_merge.len: 1917
[test_021] hits_merge.len: 1911
[test_022] hits_merge.len: 1791
[test_023] hits_merge.len: 1642
[test_024] hits_merge.len: 1898
[test_025] hits_merge.len: 1804
[test_026] hits_merge.len: 1682
[test_027] hits_merge.len: 1660
[test_028] hits_merge.len: 1816
[test_029] hits_merge.len: 1909
[test_030] hits_merge.len: 1725
[test_031] hits_merge.len: 1618
[test_03

In [6]:
query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    # 计算evidence分数
    evidence_l = []
    recall_hits= [hit for hit, _ in query_id_2_recall_hits[query_id]]
    for hit in recall_hits:
        cc = citation_utils.parse_cc_output_citations_and_sentences(hit['text'])
        for citation, first_appears_idx in cc['citations']:
            evidence = citation_utils.build_evidence(cc['sentences'], first_appears_idx, window_size=5)
            evidence_l.append({'citation':citation, 'text':evidence})

    print("===>",query_id, ", evidence_l.len:", len(evidence_l))

    raw_query = query_list[-1]
    normalized_query_list = query_list[:-1] # 最后一个包含多个问题
    question_l = []
    for q in normalized_query_list:
        _, question = q.rsplit('\n\n', 1)
        question_l.append(question)
    
    evidence_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, evidence_l)
    # evidence_with_score_l_question = reranker_utils.rerank_by_batch_chunked2(reranker, '\n\n'.join(question_l), evidence_l)
    # evidence_with_score_l = [(raw_score[0], raw_score[1]+0.5*question_score[1]) for raw_score, question_score in zip(evidence_with_score_l_raw, evidence_with_score_l_question)]
    
    evidence_with_score_l = evidence_with_score_l_raw
    
    # agg by sum
    # citation_score_d = defaultdict(float)
    # for evidence, score in evidence_with_score_l:
    #     citation_score_d[evidence['citation']] += score

    # agg by top-k score sum
    citation_evidence_score_l_d = defaultdict(list)
    for evidence, score in evidence_with_score_l:
        citation_evidence_score_l_d[evidence['citation']].append((evidence, score))

    citation_score_d = {}
    for citation, l in citation_evidence_score_l_d.items():
        l2 = sorted(l, key=lambda x: x[1], reverse=True)[:5]
        score = sum([score for _, score in l2])
        citation_score_d[citation] = score
        
    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_d.items()], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    print("query_result2:", len(query_result2))
    
    result_citation_l.append(';'.join(query_result2[:100]))
    query_id_l.append(query_id)
    
    print(f"{query_id} query_result2.len:", len(query_result2))


result_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':result_citation_l})
result_df.to_csv("../output/submission_100.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

===> test_001 , evidence_l.len: 3194


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


query_result2: 472
test_001 query_result2.len: 472
===> test_002 , evidence_l.len: 5444
query_result2: 706
test_002 query_result2.len: 706
===> test_003 , evidence_l.len: 5911
query_result2: 1000
test_003 query_result2.len: 1000
===> test_004 , evidence_l.len: 6348
query_result2: 645
test_004 query_result2.len: 645
===> test_005 , evidence_l.len: 9121
query_result2: 1403
test_005 query_result2.len: 1403
===> test_006 , evidence_l.len: 11763
query_result2: 1000
test_006 query_result2.len: 1000
===> test_007 , evidence_l.len: 6919
query_result2: 829
test_007 query_result2.len: 829
===> test_008 , evidence_l.len: 7650
query_result2: 540
test_008 query_result2.len: 540
===> test_009 , evidence_l.len: 10216
query_result2: 806
test_009 query_result2.len: 806
===> test_010 , evidence_l.len: 7713
query_result2: 443
test_010 query_result2.len: 443
===> test_011 , evidence_l.len: 6554
query_result2: 985
test_011 query_result2.len: 985
===> test_012 , evidence_l.len: 7986
query_result2: 1048
test

In [7]:
# limit=35
# sub_df = pd.read_csv("../output/submission_100.csv")
# pred_l = []
# for query_id, pred in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
#     pred2 = pred.split(";")[:limit]
#     pred_l.append(';'.join(pred2))
# new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':pred_l})
# new_sub_df.to_csv("../output/submission2.csv", index=False)

In [9]:
limit=30
cc_limit=5
predicted_citation_l = []
sub_df = pd.read_csv("../output/submission_100.csv")
for query_id, preds in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    cc_l = []
    law_l = []
    
    pred_l = preds.split(';')
    for pred in pred_l:
        if pred in court_consideration_d:
            if len(cc_l) >= cc_limit:
                pass
            else:
                cc_l.append(pred)
    for pred in pred_l:
        if limit <= len(cc_l) + len(law_l):
                break
        if pred in law_d:
            law_l.append(pred)

    citation_l = cc_l.copy()
    citation_l.extend(law_l)

    predicted_citation_l.append(';'.join(citation_l[:limit]))

new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':predicted_citation_l})
new_sub_df.to_csv("../output/submission2.csv", index=False)
    